# CredResolve — Data Analyst Investigation

This notebook starts with profiling and payment forensics. It deliberately does not claim that recovery improved by 11%; that conclusion requires reconciled payments, fixed denominators and comparable cohorts.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
DATA = Path('../data') if Path('../data').exists() else Path('data')


In [ ]:
files = sorted(DATA.glob('*.csv'))
profile = []
for f in files:
    d = pd.read_csv(f)
    profile.append({'dataset': f.name, 'rows': len(d), 'columns': len(d.columns),
                    'exact_duplicates': int(d.duplicated().sum()),
                    'missing_cells': int(d.isna().sum().sum())})
profile_df = pd.DataFrame(profile).sort_values('rows', ascending=False)
profile_df


## Event coverage check

The assignment describes approximately 12 months. Check actual event dates and report gaps explicitly.

In [ ]:
event_tables = ['calls','call_attempts','payments','promises_to_pay','whatsapp_events','sms_events','field_visits','complaints']
coverage = []
for name in event_tables:
    d = pd.read_csv(DATA / f'{name}.csv')
    ts = pd.to_datetime(d['event_at'], errors='coerce')
    coverage.append({'table': name, 'valid_timestamps': ts.notna().sum(), 'min_event': ts.min(), 'max_event': ts.max()})
pd.DataFrame(coverage)


## Payment status and duplicate-reference audit

A repeated reference is a warning, not proof of a duplicate payment. Never discard rows solely because `payment_reference` repeats.

In [ ]:
payments = pd.read_csv(DATA / 'payments.csv')
payments['amount'] = pd.to_numeric(payments['amount'], errors='coerce')
print(payments['payment_status'].value_counts(dropna=False))
ref_audit = payments.groupby('payment_reference', dropna=False).agg(
    rows=('payment_id','size'), accounts=('account_id','nunique'),
    amounts=('amount','nunique'), statuses=('payment_status','nunique'),
    amount_sum=('amount','sum')).reset_index()
ref_audit.query('rows > 1').sort_values('rows', ascending=False).head(20)


## Next decisions

Before producing a golden payment table, inspect repeated references across accounts, status transitions, reversals, provider IDs and event timestamps. Agree a net-recovery definition and keep rejected/corrected records auditable.